In [ ]:
# check the variable
!echo $WORKSPACE_BUCKET

In [ ]:
!echo ${GOOGLE_PROJECT}

In [ ]:
%run Setting_Env_Variables.ipynb

In [ ]:
%%bash
dsub --version

In [ ]:
%pip install --upgrade dsub

In [ ]:
%%bash
dsub --version

In [ ]:
!echo $ARTIFACT_REGISTRY_DOCKER_REPO

In [ ]:
%env ARTIFACT_REGISTRY_DOCKER_REPO=us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod

In [ ]:
print(os.environ.get("ARTIFACT_REGISTRY_DOCKER_REPO"))

In [ ]:
%%writefile ~/aou_dsub.bash

#!/bin/bash

# This shell function passes reasonable defaults for several dsub parameters, while
# allowing the caller to override any of them. It creates a nice folder structure within
# the workspace bucket for dsub log files.
# --[ Parameters ]--
# any valid dsub parameter flag
#--[ Returns ]--
# the job id of the job created by dsub
#--[ Details ]--
# The first eight parameters below should always be those values when running on AoU RWB.
# Note that we insert some job data into the logging path.
# https://github.com/DataBiosphere/dsub/blob/main/docs/logging.md#inserting-job-data

function aou_dsub () {

  # Get a shorter username to leave more characters for the job name.
  local DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1 | tr '.' '-' | tr -d '\r')"
  

  dsub \
      --provider google-batch \
      --user-project "${GOOGLE_PROJECT}" \
      --project "${GOOGLE_PROJECT}" \
      --network "global/networks/network" \
      --subnetwork "regions/us-central1/subnetworks/subnetwork" \
      --service-account "$(gcloud config get-value account)" \
      --use-private-address \
      --regions us-central1 \
      --user "${DSUB_USER_NAME}" \
      --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest" \
      --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
      "$@"
}

In [ ]:
%%bash
echo source ~/aou_dsub.bash >> ~/.bashrc

In [ ]:
import os
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')
%env USER_NAME={USER_NAME}

In [ ]:
!echo ${WORKSPACE_BUCKET}

In [ ]:
!sed -i '1i export DSUB_CLOUD_SDK_IMAGE="gcr.io/google.com/cloudsdktool/cloud-sdk:591.0.0-slim"' ~/aou_dsub.bash

In [ ]:
cat  ~/aou_dsub.bash